# A LangChain agent end to end

Booster LEAP, Agentic AI, topic 5. Run a prompt template with a parser, a retrieval chain and an agent built with `create_agent` (tools, middleware, traces), change them, and (optionally) give the agent a real model with your own key.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use this notebook:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again; the experiments tell you what to try.

## Stage 1: the program with stand-in models

Every model in stage 1 is a **stand-in**: a fake chat model from `langchain_core` that returns prepared replies. It replaces the *model* only. The prompts, the parser, the retriever, the tools, the middleware and the agent are the real LangChain objects, so the traces printed below are real. Run the next cell once; it installs the package.

In [ ]:
# Installs the LangChain package (the outputs below were produced with langchain 1.5.2, langchain-core 1.8.4 and langgraph 1.2.24;
# a later release can differ in small details). This also installs langchain-core and langgraph.
%pip install -q langchain

The setup cell: imports, the support desk data (orders with owners, four policy passages), a small retriever that ranks by shared words, the stand-in model, two tools (`lookup_order` checks that the order belongs to the logged-in customer; `search_policy` is the retriever as a tool) and a middleware function that turns a failing tool into text. It prints nothing.

In [ ]:
import json
import re

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, wrap_tool_call
from langchain_core.documents import Document
from langchain_core.language_models.fake_chat_models import FakeListChatModel, GenericFakeChatModel
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import ToolException, tool

CUSTOMER = "C1"                       # the customer who is logged in (topic 2: identity comes from the session, not from the model)
ORDERS = {"A1001": {"owner": "C1", "status": "shipped"}, "A1002": {"owner": "C2", "status": "delivered"}, "A1003": {"owner": "C1", "status": "processing"}}
DOCS = [Document(page_content="Refunds are allowed within 30 days of delivery, for orders in the status delivered.", metadata={"id": "p1"}),
        Document(page_content="An approved refund reaches the customer's account in 5 working days.", metadata={"id": "p2"}),
        Document(page_content="An order in the status processing can be cancelled free of charge until it ships.", metadata={"id": "p3"}),
        Document(page_content="Shipped orders arrive in 3 to 7 days; the carrier sends a tracking number by email.", metadata={"id": "p4"})]
FILLER = {"a", "an", "the", "to", "do", "does", "how", "long", "is", "can", "in", "of", "for", "it", "i", "until"}

class OverlapRetriever(BaseRetriever):
    """Lesson 6: rank the passages by the words they share with the query (a small stand-in for a real index)."""
    k: int = 2
    def _get_relevant_documents(self, query, *, run_manager=None):
        q = set(re.findall(r"[a-z0-9]+", query.lower())) - FILLER
        scored = sorted(((len(q & set(re.findall(r"[a-z0-9]+", d.page_content.lower()))), d.metadata["id"], d) for d in DOCS), key=lambda s: (-s[0], s[1]))
        return [d for n, _, d in scored[: self.k] if n >= 2]

retriever = OverlapRetriever(k=2)

def format_docs(docs):
    return "\n".join(f"[{d.metadata['id']}] {d.page_content}" for d in docs) or "(no passages found)"

class StandInModel(GenericFakeChatModel):
    """A stand-in for the MODEL only: it returns the prepared messages in order and ignores its input. bind_tools returns itself."""
    def bind_tools(self, tools, **kwargs):
        return self

CALLS = []

def call(name, **args):
    """A prepared model reply that asks for a tool call. Each call builds a NEW message: the state keeps one message per message id, so one message object repeated in a list would not grow the history."""
    CALLS.append(name)
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": f"call_{len(CALLS)}"}])

@tool
def lookup_order(order_id: str) -> str:
    """Find one order of the logged-in customer by its id and return its status."""
    order = ORDERS.get(order_id)
    if order is None or order["owner"] != CUSTOMER:                       # the ownership rule is yours, not the framework's
        raise ToolException(f"no order {order_id} on this account; ids look like A1001")
    return f"order {order_id}: {order['status']}"

@tool
def search_policy(query: str) -> str:
    """Search the support policy and return the matching passages with their ids."""
    return format_docs(retriever.invoke(query))

@wrap_tool_call
def tool_errors(request, handler):
    """Middleware (lesson 5): a failing tool becomes a message the model can read, instead of ending the run."""
    try:
        return handler(request)
    except Exception as error:
        return ToolMessage(content=f"tool error: {error}", tool_call_id=request.tool_call["id"])

Two chains with a fixed order of steps. Part 1: template, model, parser turn a ticket into a dictionary. Part 2: a retrieval chain finds passages, builds a prompt with ids and returns an answer; a short check tests that the cited id was among the retrieved passages.

In [ ]:
# 1. prompt template + model + parser: a ticket becomes a dictionary (topic 5, lessons 1 to 3)
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Return only JSON with the keys order_id and wish. wish is one of cancel, refund, status."),
    ("human", "{ticket}")])
extract_model = FakeListChatModel(responses=['{"order_id": "A1003", "wish": "cancel"}'])          # a stand-in for the model
extract = extract_prompt | extract_model | JsonOutputParser()
fields = extract.invoke({"ticket": "Hello, I am C1. Please cancel my order A1003, it is still processing."})
print("1. extracted:", fields)

# 2. a retrieval chain: the search is a fixed step (lesson 6)
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the passages. Cite the id of each passage you use in square brackets. If they do not answer, say you cannot answer.\n\nPassages:\n{context}"),
    ("human", "{question}")])
answer_model = FakeListChatModel(responses=["An order in the status processing can be cancelled free of charge until it ships [p3]."])
rag = {"context": retriever | format_docs, "question": RunnablePassthrough()} | answer_prompt | answer_model | StrOutputParser()
question = "Can I cancel an order in the status processing for free?"
shown = [d.metadata["id"] for d in retriever.invoke(question)]
reply = rag.invoke(question)
cited = re.findall(r"\[(p\d+)\]", reply)
print("2. retrieved:", shown, "| reply:", reply)
print("   citation check:", "ok" if cited and set(cited) <= set(shown) else "PROBLEM")

1. extracted: {'order_id': 'A1003', 'wish': 'cancel'}
2. retrieved: ['p3'] | reply: An order in the status processing can be cancelled free of charge until it ships [p3].
   citation check: ok


The agent. `make_agent` builds an agent with `create_agent`, the two tools and the middleware; `trace` runs it and prints every message of the state. Run 3a is a normal run (the policy search is a tool the model chooses to call). Run 3b meets a tool error that the middleware turns into text. Run 3c has a model that never stops, ended by a call limit.

In [ ]:
# 3. an agent: tools (one of them a retrieval step the model may choose), middleware, a trace (lessons 4 and 5)
def make_agent(replies, call_limit=6):
    return create_agent(StandInModel(messages=iter(replies)), tools=[lookup_order, search_policy],
                        system_prompt="You are the support desk assistant. Use the tools; cite policy ids.",
                        middleware=[tool_errors, ModelCallLimitMiddleware(run_limit=call_limit, exit_behavior="end")])

def trace(agent, question):
    state = agent.invoke({"messages": [{"role": "user", "content": question}]})
    for m in state["messages"]:
        kind = type(m).__name__.replace("Message", "")
        text = f"CALL {m.tool_calls[0]['name']}({m.tool_calls[0]['args']})" if getattr(m, "tool_calls", None) else m.content
        print(f"   {kind:5} {text}")
    return state

print("3a. a normal run")
trace(make_agent([call("lookup_order", order_id="A1003"), call("search_policy", query="cancel order processing free"),
                  AIMessage(content="A1003 is processing, so you can cancel it free of charge [p3].")]),
      "Where is A1003, and can I cancel it for free?")
print("3b. a tool error that the middleware turns into text")
trace(make_agent([call("lookup_order", order_id="A1002"), AIMessage(content="I cannot find A1002 on your account.")]),
      "Where is A1002?")
print("3c. a model that keeps calling the same tool, stopped by a call limit")
state = trace(make_agent([call("lookup_order", order_id="A1001") for _ in range(20)], call_limit=2), "Where is A1001?")
print("   messages:", len(state["messages"]))

3a. a normal run
   Human Where is A1003, and can I cancel it for free?
   AI    CALL lookup_order({'order_id': 'A1003'})
   Tool  order A1003: processing
   AI    CALL search_policy({'query': 'cancel order processing free'})
   Tool  [p3] An order in the status processing can be cancelled free of charge until it ships.
   AI    A1003 is processing, so you can cancel it free of charge [p3].
3b. a tool error that the middleware turns into text
   Human Where is A1002?
   AI    CALL lookup_order({'order_id': 'A1002'})
   Tool  tool error: no order A1002 on this account; ids look like A1001
   AI    I cannot find A1002 on your account.
3c. a model that keeps calling the same tool, stopped by a call limit
   Human Where is A1001?
   AI    CALL lookup_order({'order_id': 'A1001'})
   Tool  order A1001: shipped
   AI    CALL lookup_order({'order_id': 'A1001'})
   Tool  order A1001: shipped
   AI    Model call limits exceeded: run limit (2/2)
   messages: 6


## Experiments (stage 1)

Change one thing at a time, predict the output, then run the cell again (run the setup cell first when you change it).

1. Change `call_limit=2` in run 3c to `call_limit=4`. How many messages does the final state hold?
2. Set `CUSTOMER = "C2"` in the setup cell and run 3b again. What does the tool return for A1002? Which orders are refused now?
3. Remove `tool_errors` from the middleware list in `make_agent`. What happens to run 3b?
4. Change the reply of `extract_model` in part 1 to `'Sure! {"order_id": '`. Which error ends the cell, and where would a repair step go?
5. Call `search_policy.invoke({"query": "ship orders to other countries"})`. What does it return, and what should the agent say next?
6. In run 3c replace the list comprehension by one repeated message: `[call("lookup_order", order_id="A1001")] * 20`. How many messages does the state hold now, and why?

## Stage 2 (optional): a real model

Stage 1 used stand-ins. This stage gives the same agent a real model with **your own key**. Read before you run:

- It sends the question, the tool results and the policy passages to the provider and costs a small amount of money. Each run is a few short calls, but you pay for what you run.
- Never write the key into a cell or share a notebook that contains it. The cell reads it from Colab's Secrets.
- A real model chooses its own tool calls, so the trace differs from run to run and from the stand-in. The tools, the ownership rule, the middleware and the call limit are the same code.

In [ ]:
# Optional stage 2: the stand-in model is replaced by a real model. This sends the conversation and the tool results to the provider and costs a little money.
# One-time set-up: open the key symbol (Secrets) in the left bar of Colab, add a secret named OPENAI_API_KEY and switch "Notebook access" on.
%pip install -q "langchain[openai]"
import os
from google.colab import userdata
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
MODEL = "PUT-A-MODEL-NAME-HERE"       # a chat model of your account (see the provider's documentation)
assert MODEL != "PUT-A-MODEL-NAME-HERE", "choose a model first"
real_model = init_chat_model(MODEL, model_provider="openai")


def make_real_agent(call_limit=6):
    return create_agent(real_model, tools=[lookup_order, search_policy],
                        system_prompt="You are the support desk assistant. Use the tools; cite policy ids.",
                        middleware=[tool_errors, ModelCallLimitMiddleware(run_limit=call_limit, exit_behavior="end")])

In [ ]:
for question in ["Where is A1003, and can I cancel it for free?", "Where is A1002?", "Do you ship orders to other countries?"]:
    print("question:", question)
    trace(make_real_agent(), question)

## What to look at

1. Did the real model call `lookup_order` and `search_policy` in the same order as the stand-in? Did it call anything you did not expect?
2. For A1002 (not the customer's order): what did the model say after the tool error text? Did it reveal anything about the order?
3. For the question about shipping abroad: did it search, and did it say the policy does not cover it, or did it answer from its own knowledge?
4. Did every policy statement carry an id such as `[p3]`, and was that id among the passages the tool returned?
5. Write down one case where the real model and the stand-in differ, and which part of the program protected you.